# Experimentation notebook for hybrid CNNs

## Dependencies

In [1]:
import os, sys, time, copy, argparse
from pathlib import Path

import numpy as np
import pandas as pd
import scipy as sp
import scipy.sparse as sps
import torch
import neuropythy as ny

import matplotlib as mpl
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader


sys.path.append(str(Path.home() / 'visual-autolabel' / 'src'))

import visual_autolabel as val

sys.path.append(str(Path.home() / 'volume-cnn' / 'src'))

import volcnn

from volcnn import (
    HCPVolumeDataset,
    make_dataloaders,
    sids,
    data_cache_path,
    dice_loss,
    bce_loss,
    UNet3D)


/home/leo/.conda/envs/cnn/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/leo/.conda/envs/cnn/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


## Configuration

In [2]:
torch.__version__

'2.5.1'

In [3]:
dataset2D_cache_path = '/data/visual-autolabel/datasets/HCP'

val.image.dataset3D_cache_path = '/data/visual-autolabel/volumetric/data'
val.image.noddi_data_path = '/data/visual-autolabel/volumetric/NODDI'

inputs3D = ('graymask', 'T1', 'T2')
inputs2D = ('curvature', 'convexity', 'thickness')
outputs = ('V1', 'V2', 'V3')
batch_size = 1
lr = 0.0075
gamma = 0.95
num_epochs = 30
zoom = 1/2
subindex = (slice(2,-2), slice(8, 256+8), slice(2,-2))
dtype = torch.float32
device = 'cpu'
shuffle = True

In [4]:
!uname -a

Linux tapas 5.4.0-150-generic #167~18.04.1-Ubuntu SMP Wed May 24 00:51:42 UTC 2023 x86_64 x86_64 x86_64 GNU/Linux


## Setting up Training Loop

In [5]:
# Training and validation subjects
trn_sids = [100610, 118225, 140117, 158136, 197348, 214524, 346137, 412528,
            573249, 724446, 905147, 102311, 159239, 173334, 221319, 352738,
            429040, 725751, 826353, 910241, 102816, 145834, 162935, 175237,
            199655, 233326, 436845, 732243, 926862, 104416, 128935, 146129,
            164131, 200210, 365343, 463040, 751550, 859671, 927359, 105923,
            130114, 146432, 164636, 187345, 200311, 467351, 617748, 757764,
            942658, 108323, 130518, 165436, 191033, 200614, 249947, 381038,
            525541, 627549, 109123, 131217, 146937, 167036, 177746, 191336,
            201515, 385046, 536647, 638049, 770352, 872764, 111312, 167440,
            178142, 191841, 203418, 257845, 541943, 771354, 878776, 958976,
            111514, 132118, 169040, 178243, 393247, 547046, 654552, 878877,
            966975, 114823, 155938, 205220, 283543, 395756, 550439, 671855,
            783462, 898176, 971160, 156334, 180533, 193845, 318637, 397760,
            552241, 680957, 899885, 973770, 115825, 135124, 157336, 169747,
            181232, 401422, 562345, 690152, 814649, 901139, 995174, 116726,
            137128, 158035, 181636, 196144, 330324, 406836, 572045, 818859]
val_sids = [765864, 209228, 134829, 585256, 901442, 169444, 380036, 389357,
            581450, 198653, 115017, 782561, 176542, 246133, 185442, 601127,
            204521, 195041, 182739, 212419, 263436, 320826, 825048, 192641,
            360030, 177140, 146735, 126426, 789373, 871762, 172130, 171633]

In [6]:
trn_dataset3D = val.image.HCPDataset3D(
    sids=trn_sids,
    inputs=inputs3D,
    outputs=outputs,
    cache_path=val.image.dataset3D_cache_path,
    dtype=dtype,
    device=device,
    mkdir_mode=0o775,
    subindex=subindex,
    zoom=zoom)

In [7]:
trn_dataset2D = val.benson2025.hcp.HCPDataset(
    inputs2D, outputs,
    sids=trn_sids)

In [8]:
class VolumeToFlatImageDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 cache_path='/data/visual-autolabel/volumetric/data/affines'):
        self.cache_path = Path(cache_path)
        self.sids = sids
        self.data = {}
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        sid = self.sids[k]
        if sid in self.data:
            return self.data[sid]
        matrix = torch.load(f"{self.cache_path}/{sid}.pt")
        import scipy.sparse as sps
        (row, col, val) = sps.find(matrix)
        matrix = torch.sparse_csr_tensor(
            torch.as_tensor(row), torch.as_tensor(col), 
            torch.as_tensor(val),
            matrix.shape)
        matrix = torch.tensor(matrix, dtype=torch.float32)
        self.data[sid] = matrix
        return matrix

In [9]:
class HCPHybridDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 inputs2D,
                 inputs3D,
                 outputs=('V1', 'V2', 'V3'),
                 cache_path2D=None,
                 cache_path3D=None,
                 transform_cache_path='/data/visual-autolabel/volumetric/data/affines',
                 dtype=None,
                 device=None,
                 mkdir_mode=509,
                 subindex=(slice(2, -2, None), slice(8, 264, None), slice(2, -2, None)),
                 zoom=0.5,
                 ):
        self.transform_dataset = VolumeToFlatImageDataset(sids, cache_path=transform_cache_path)
        self.dataset3D = val.image.HCPDataset3D(
            sids=sids,
            inputs=inputs3D,
            outputs=outputs,
            cache_path=cache_path3D,
            dtype=dtype,
            device=device,
            mkdir_mode=0o775,
            subindex=subindex,
            zoom=zoom)
        self.dataset2D = val.benson2025.hcp.HCPDataset(
            inputs2D, 
            outputs,
            sids=sids,
            cache_path=cache_path2D)
        self.sids = sids
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        inputdata3D, _ = self.dataset3D[k]
        transformdata3D = self.transform_dataset[k]
        inputdata2D, outputdata2D = self.dataset2D[k]
        return (inputdata3D, transformdata3D, inputdata2D, outputdata2D)
        

In [10]:
model = val.image.UNet(len(inputs3D), len(inputs3D), len(inputs2D), len(outputs))
inputs3D

('graymask', 'T1', 'T2')

In [11]:
trn_transforms = VolumeToFlatImageDataset(trn_sids)

In [ ]:

# Define custom collate function to handle sparse matrices
def hybrid_collate(batch):
    inputs3D, transforms, inputs2D, labels = zip(*batch)
    return (
        torch.stack(inputs3D),          # [B, C, D, H, W]
        list(transforms),              # keep sparse transforms as list
        torch.stack(inputs2D),         # [B, C, H, W]
        torch.stack(labels)            # [B, C, H, W]
    )

# Training and validation subjects
trn_sids = [100610, 118225]
val_sids = [765864, 209228]


trn_dataset = HCPHybridDataset(
    sids=trn_sids,
    inputs2D=inputs2D,
    inputs3D=inputs3D,
    outputs=outputs,
    cache_path2D=dataset2D_cache_path,
    cache_path3D=val.image.dataset3D_cache_path,
    transform_cache_path='/data/visual-autolabel/volumetric/data/affines',
    dtype=dtype,
    device=device,
    mkdir_mode=0o775,
    subindex=subindex,
    zoom=zoom
)

val_dataset = HCPHybridDataset(
    sids=val_sids,
    inputs2D=inputs2D,
    inputs3D=inputs3D,
    outputs=outputs,
    cache_path2D=dataset2D_cache_path,
    cache_path3D=val.image.dataset3D_cache_path,
    transform_cache_path='/data/visual-autolabel/volumetric/data/affines',
    dtype=dtype,
    device=device,
    mkdir_mode=0o775,
    subindex=subindex,
    zoom=zoom
)

# Create DataLoaders with custom collate
train_loader = DataLoader(trn_dataset, batch_size=batch_size, shuffle=True, collate_fn=hybrid_collate)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=hybrid_collate)
dataloaders = {'trn': train_loader, 'val': val_loader}

# Initialize model
model = val.image.UNet(len(inputs3D), len(inputs3D), len(inputs2D), len(outputs))
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=gamma)

# Training loop
best_loss = np.inf
best_weights = None
print("Starting training...")

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1:02d}/{num_epochs}")
    losses_epoch = {'trn': [], 'val': []}
    dice_losses_epoch = {'trn': [], 'val': []}
    wd = epoch / (num_epochs - 1)
    wb = 1 - wd

    for phase in ['trn', 'val']:
        print('  Training..' if phase == 'trn' else '  Testing..', end='')
        model.train() if phase == 'trn' else model.eval()
        running_loss = 0.0
        running_dice = 0.0
        count = 0

        for inputs3D, transforms, inputs2D, labels in dataloaders[phase]:
            inputs3D, inputs2D, labels = inputs3D.to(device), inputs2D.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'trn'):
                outputs = model(inputs3D, inputs2D, transforms)
                loss_bce = bce_loss(outputs, labels)
                loss_dice = dice_loss(outputs, labels)
                loss = wb * loss_bce + wd * loss_dice

                if phase == 'trn':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs3D.size(0)
            running_dice += loss_dice.item() * inputs3D.size(0)
            count += inputs3D.size(0)

        epoch_loss = running_loss / count
        epoch_dice = running_dice / count

        print(f"  {phase} loss: {epoch_loss:.4f}, dice: {epoch_dice:.4f}")

        if phase == 'val' and epoch_dice < best_loss:
            best_loss = epoch_dice
            best_weights = copy.deepcopy(model.state_dict())

    scheduler.step()

# Save best model
model.load_state_dict(best_weights)
torch.save(model.state_dict(), Path.home() / 'bestmodel_hybrid.pt')
print("Training complete. Best dice loss:", best_loss)

Starting training...
Epoch 01/30
  Training..

/home/leo/visual-autolabel/src/visual_autolabel/image/_data3D.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(filepath)
/tmp/ipykernel_28783/329577022

In [ ]:
model = val.image.UNet(
    len(inputs3D),
    len(inputs3D),
    len(inputs2D),
    len(outputs))

In [ ]:
k = 0

(features3D, _) = trn_dataset3D[k]
(features2D, labels) = trn_dataset2D[k]
transform = trn_transforms[k]

outputs = model(features3D[None,...], features2D[None,...], transform)

In [ ]:
# This should look wrong since we're running an untrained model, but if it produces
# an image with a width of 256 and a height of 128, things are working!
probs = torch.sigmoid(outputs)  # Convert from logits to probabilities.
im = np.transpose(probs.detach().numpy()[0], (1,2,0))
plt.imshow(im)